#Code Adapted From https://github.com/kundajelab/chrombpnet-figures/tree/main/src/figures_factory/Fig1

#base environment
#pip install pyBigwig

In [ ]:
import h5py
import pyBigWig
import matplotlib
from tqdm import tqdm
import scipy.ndimage
import numpy as np
import matplotlib
from matplotlib import pyplot as plt
from matplotlib import cm
from matplotlib.colors import Normalize 
from scipy.interpolate import interpn
import math
import pandas as pd
import json 

matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42 

In [ ]:
file1= "~/chrombpnet/chrombpnet_use_snakefile/Treeshrew/TS_C8Epithelial_cells/output/models/fold_0/C8.1/evaluation/chrombpnet_predictions.h5"
bigwig="~/chrombpnet/chrombpnet_use_snakefile/Treeshrew/TS_C8Epithelial_cells/data/cut_site_bigwig/C8.1_unstranded.bw"
#peaks="~/Human_embryo_project_T250614/FileSave/1000bp_OBO/modified_peak/C8.1_W1000bp_Peak.modified.bed"
#nonpeaks="~/chrombpnet/chrombpnet_use_snakefile/Treeshrew/TS_C8Epithelial_cells/data/nonpeaks/fold_0/C8.1_negatives.bed"

chr_fold_path="~/chrombpnet/old/chrombpnet_TS_Mono/fold_split_version2/data/splits/fold_0.json"
outf="~/chrombpnet/chrombpnet_use_snakefile/Treeshrew/TS_C8Epithelial_cells/output/predict/C8.1"


# Plotting

In [ ]:
def density_scatter(x, y, xlab, ylab, ax = None, sort = True, bins = 20):
    """
    Scatter plot colored by 2d histogram
    """
    bad_indices=np.where(np.isnan(x))+np.where(np.isnan(y))
    x=x[~np.isin(np.arange(x.size),bad_indices)]
    y=y[~np.isin(np.arange(y.size),bad_indices)]

    if ax is None :
        fig , ax = plt.subplots()
    data , x_e, y_e = np.histogram2d( x, y, bins = bins, density = True )
    z = interpn( ( 0.5*(x_e[1:] + x_e[:-1]) , 0.5*(y_e[1:]+y_e[:-1]) ) , data , np.vstack([x,y]).T , method = "splinef2d", bounds_error = False)

    #To be sure to plot all data
    z[np.where(np.isnan(z))] = 0.0
    # Sort the points by density, so that the densest points are plotted last
    if sort :
        idx = z.argsort()
        x, y, z = x[idx], y[idx], z[idx]

    #z = [plt.cm.viridis(t) for t in z]
    ax.scatter( x, y, c=z ,linewidths=0,edgecolors=None, rasterized=True)
    #ax.plot(x,y,'o',rasterized=True,linewidths=0,c=z)

    norm = Normalize(vmin = np.min(z), vmax = np.max(z))
    cbar = fig.colorbar(cm.ScalarMappable(norm = norm), ax=ax)
    cbar.ax.set_ylabel('Density')
    plt.xlabel(xlab)
    plt.ylabel(ylab)
    return ax

# Main

In [ ]:
import h5py

with h5py.File(file1, "r") as data:
    def show(name, obj):
        if hasattr(obj, "shape"):
            print(name, obj.shape, obj.dtype)
        else:
            print(name)
            
    data.visititems(show)

In [ ]:
data = h5py.File(file1)

'''
mode="test"
splits_dict=json.load(open(chr_fold_path))
chroms_to_keep=splits_dict[mode]

NARROWPEAK_SCHEMA = ["chr", "start", "end", "1", "2", "3", "4", "5", "6", "summit"]
peaks_bed_regions=pd.read_csv(peaks,header=None,sep='\t',names=NARROWPEAK_SCHEMA)
peaks_bed_regions_to_keep=peaks_bed_regions[peaks_bed_regions["chr"].isin(chroms_to_keep)]

NARROWPEAK_SCHEMA = ["chr", "start", "end", "1", "2", "3", "4", "5", "6", "summit"]
nonpeaks_bed_regions=pd.read_csv(nonpeaks,header=None,sep='\t',names=NARROWPEAK_SCHEMA)
nonpeaks_bed_regions_to_keep=nonpeaks_bed_regions[nonpeaks_bed_regions["chr"].isin(chroms_to_keep)]
'''


profile_predictions = data["predictions"]["logcounts"][:]
prof_coords_chr = data["coords"]["coords_chrom"][:]
prof_coords_center = data["coords"]["coords_center"][:]
prof_coords_peak = data["coords"]["coords_peak"][:]


In [ ]:
profile_predictions

In [ ]:
prof_coords_chr[1] #bytes 类型

In [ ]:
prof_coords_chr = np.array([
    x.decode("utf-8") if isinstance(x, bytes) else x
    for x in prof_coords_chr
])
prof_coords_chr

In [ ]:
prof_coords_chr[1]

In [ ]:
prof_coords_center

In [ ]:
prof_coords_center[1]

In [ ]:
prof_coords_peak #coords_peak 本身就是 ChromBPNet 预测文件里保存的 peak/nonpeak 标签，0 表示 nonpeak，1 表示 peak

In [ ]:
prof_coords_peak[1]

In [ ]:
n_peaks = np.sum(prof_coords_peak != 0)
n_nonpeaks = np.sum(prof_coords_peak == 0)

print("n_peaks:", n_peaks)
print("n_nonpeaks:", n_nonpeaks)

print("前 n_peaks 个是否全是 peak:")
print(np.all(prof_coords_peak[:n_peaks] != 0))

print("后 n_nonpeaks 个是否全是 nonpeak:")
print(np.all(prof_coords_peak[n_peaks:] == 0))

In [ ]:
bw = pyBigWig.open(bigwig) 

true_counts_mat_peaks = []
true_counts_mat_peaks_non_peaks = []

#idx=0 #idx 专门用来数 nonpeak 文件中的第几个 nonpeak
for jdx in tqdm(range(prof_coords_chr.shape[0])): #jdx 是在循环 chrombpnet_predictions.h5 里的所有预测样本
        if prof_coords_peak[jdx] == 0: #取一个 nonpeak 区域的坐标，并计算这个区域 summit 周围 1000 bp 的窗口
                chr1 = prof_coords_chr[jdx]
                start = prof_coords_center[jdx]  - 500
                end  =  prof_coords_center[jdx]  + 500

                true_counts = np.nan_to_num(bw.values(chr1,start,end )) #bw.values(chr1, start, end) 返回的是一个长度为 1000 的数组，每一个数字表示这个碱基位置上的真实 cut-site / coverage signal，np.nan_to_num() 的作用是把 NaN 变成 0
                true_counts_mat_peaks_non_peaks.append(np.log(np.sum(true_counts)+1)) #np.sum(true_counts)把这个 1000 bp 区间所有位置的信号加起来
                
                #idx += 1
                continue

        #取一个 peak 区域的坐标，并计算这个区域 summit 周围 1000 bp 的窗口
        chr1 = prof_coords_chr[jdx]
        start = prof_coords_center[jdx]  - 500
        end  =  prof_coords_center[jdx]  + 500

        true_counts = np.nan_to_num(bw.values(chr1,start,end ))
        true_counts_mat_peaks.append(np.log(np.sum(true_counts)+1))
        

In [ ]:
import scipy.stats
cta = scipy.stats.pearsonr(true_counts_mat_peaks,profile_predictions[prof_coords_peak==1])
print(cta[1])

In [ ]:
cta

In [ ]:
#log10(p-value)
np.log10(cta[1])

In [ ]:
#p-value 太小了,下溢成 0 / -inf,可以用 mpmath 高精度计算 log10(p-value)
import numpy as np
import scipy.stats
import mpmath as mp

x = np.asarray(true_counts_mat_peaks).squeeze()
y = np.asarray(profile_predictions[prof_coords_peak == 1]).squeeze()

r, p = scipy.stats.pearsonr(x, y)

n = len(x)
df = n - 2

# 设置高精度
mp.mp.dps = 100

r_mp = mp.mpf(str(r))
df_mp = mp.mpf(str(df))

# Pearson r 对应的 t statistic
t_mp = r_mp * mp.sqrt(df_mp / (1 - r_mp**2))

# two-sided p-value for t distribution:
# p = I_{df/(df+t^2)}(df/2, 1/2)
x_beta = df_mp / (df_mp + t_mp**2)

p_mp = mp.betainc(
    df_mp / 2,
    mp.mpf("0.5"),
    0,
    x_beta,
    regularized=True
)

log10_p = mp.log10(p_mp)

print(f"n = {n}")
print(f"Pearson r = {r:.4f}")
print(f"scipy p-value = {p:.3e}")
print("log10(p-value) =", log10_p)
print("p-value ≈ 10^", log10_p)

In [ ]:
plt.rcParams["figure.figsize"]=8,8
fig=plt.figure() 
density_scatter(np.array(true_counts_mat_peaks),
                profile_predictions[prof_coords_peak==1],
                xlab='Log Count Labels',
                ylab='Log Count Predictions')

#plt.savefig(outf + "/K562.scatter.plot.peaks.pdf",format='pdf',dpi=300, transparent=True)
plt.savefig("~/chrombpnet/chrombpnet_use_snakefile/Treeshrew/TS_C8Epithelial_cells/output/predict/C8.1/C8.1.scatter.plot.peaks.pdf",format='pdf',dpi=300, transparent=True)


In [ ]:
file1= "~/chrombpnet/chrombpnet_use_snakefile/Treeshrew/TS_C8Epithelial_cells/output/models/fold_0/C8.1/evaluation/chrombpnet_predictions.h5"
bigwig="~/chrombpnet/chrombpnet_use_snakefile/Treeshrew/TS_C8Epithelial_cells/data/cut_site_bigwig/C8.1_unstranded.bw"
rep1=pyBigWig.open("~/chrombpnet/chrombpnet_use_snakefile/Treeshrew/TS_C8Epithelial_cells/data/cut_site_bigwig/C8.1.pr1_unstranded.bw")
rep2=pyBigWig.open("~/chrombpnet/chrombpnet_use_snakefile/Treeshrew/TS_C8Epithelial_cells/data/cut_site_bigwig/C8.1.pr2_unstranded.bw")
data = h5py.File(file1)

import pandas as pd

#peaks="~/Human_embryo_project_T250614/FileSave/1000bp_OBO/modified_peak/C8.1_W1000bp_Peak.modified.bed"
chr_fold_path="~/chrombpnet/old/chrombpnet_TS_Mono/fold_split_version2/data/splits/fold_0.json"

In [ ]:
'''
NARROWPEAK_SCHEMA = ["chr", "start", "end", "1", "2", "3", "4", "5", "6", "summit"]
bed_regions=pd.read_csv(peaks,header=None,sep='\t',names=NARROWPEAK_SCHEMA)
mode="test"
splits_dict=json.load(open(chr_fold_path))
chroms_to_keep=splits_dict[mode]

bed_regions_to_keep=bed_regions[bed_regions["chr"].isin(chroms_to_keep)]
print(bed_regions_to_keep.shape)
print(bed_regions_to_keep.head())
'''

profile_predictions = data["predictions"]["profs"][:]
prof_coords_chr = data["coords"]["coords_chrom"][:]
prof_coords_center = data["coords"]["coords_center"][:]
prof_coords_peak = data["coords"]["coords_peak"][:]
num_bins=20
plt.rcParams["figure.figsize"]=8,8

In [ ]:
prof_coords_chr = np.array([
    x.decode("utf-8") if isinstance(x, bytes) else x
    for x in prof_coords_chr
])
prof_coords_chr

In [ ]:
#print(prof_coords_chr.shape)
jsd_pw=[]
jsd_rnd=[]
jsd_rep=[]
jsd_ml=[]

In [ ]:
#jdx=0
bw = pyBigWig.open(bigwig) 
pseudocount=0.001

true_counts_mat = []
rep1_counts_mat = []
rep2_counts_mat = []
for idx in tqdm(range(prof_coords_chr.shape[0])):
	if prof_coords_peak[idx] == 0:
		continue

	chr = prof_coords_chr[idx]
	start = prof_coords_center[idx]  - 500
	end  =  prof_coords_center[idx]  + 500

	true_counts = np.nan_to_num(bw.values(chr,start,end ))+pseudocount
	true_counts_mat.append(true_counts)

	rep1_counts_mat.append(np.nan_to_num(rep1.values(chr,start,end ))+pseudocount)
	rep2_counts_mat.append(np.nan_to_num(rep2.values(chr,start,end ))+pseudocount)

	#jdx+=1


mean_profile_label=np.nanmean(true_counts_mat,axis=0) #对所有 peak 的真实 observed profile 求平均（沿着 peak 这个方向 求平均，也就是每一个位置单独求平均），得到一个“平均 observed profile”，用于比较Average Observed vs Observed

In [ ]:
from scipy.spatial.distance import jensenshannon

In [ ]:
jdx=0
for idx in tqdm(range(prof_coords_chr.shape[0])):
	if prof_coords_peak[idx] == 0:
		continue

	true_counts =  true_counts_mat[jdx]

	shuffled_labels=np.random.permutation(true_counts) #把当前这个 peak 的真实 observed profile 中 1000 个位置的 signal 随机打乱顺序
	pred_probs = profile_predictions[idx]


	#true_counts = scipy.ndimage.gaussian_filter1d(true_counts, 7,axis=0, truncate=(80 / 14))
	#pred_probs = scipy.ndimage.gaussian_filter1d(pred_probs, 7,axis=0, truncate=(80 / 14))

	cur_jsd=jensenshannon(true_counts/(np.nansum(true_counts)),pred_probs) #true_counts/(np.nansum(true_counts))把真实 signal 转换成 概率分布
	jsd_pw.append(cur_jsd)

	rep1_counts = rep1_counts_mat[jdx]
	rep2_counts = rep2_counts_mat[jdx]

	mean_profile_label_prob=mean_profile_label/np.nansum(mean_profile_label)

	cur_mean_lbl=jensenshannon(true_counts/(np.nansum(true_counts)),mean_profile_label_prob)
	jsd_ml.append(cur_mean_lbl)


	#plt.figure()
	#plt.plot(true_counts/(np.nansum(true_counts)))
	#plt.plot(pred_probs)
	#plt.show()

	#shuffled_labels = scipy.ndimage.gaussian_filter1d(shuffled_labels, 7,axis=0, truncate=(80 / 14))
	shuffled_labels_prob=shuffled_labels/(np.nansum(shuffled_labels))


	rep1_counts = scipy.ndimage.gaussian_filter1d(rep1_counts, 7,axis=0, truncate=(80 / 14)) #对 pseudoreplicate 的 observed profile 做 Gaussian smoothing 平滑，降低 pseudoreplicate profile 的随机噪声，让 rep1 和 rep2 的比较更稳定
	rep2_counts = scipy.ndimage.gaussian_filter1d(rep2_counts, 7,axis=0, truncate=(80 / 14))
	curr_jsd_rnd=jensenshannon(true_counts/(np.nansum(true_counts)),shuffled_labels_prob)
	jsd_rnd.append(curr_jsd_rnd)

	rep_jsd=jensenshannon(rep1_counts/(np.nansum(rep1_counts)),rep2_counts/(np.nansum(rep2_counts)))
	jsd_rep.append(rep_jsd)

	jdx+=1

In [ ]:
print(np.nanmedian(jsd_pw))
print(np.nanmedian(jsd_rnd))
print(np.nanmedian(jsd_rep))

In [ ]:
#print(jsd_pw)
plt.figure()
n,bins,patches=plt.hist(jsd_pw,num_bins,facecolor='blue',alpha=0.5,label="Predicted vs Observed")
n1,bins1,patches1=plt.hist(jsd_rnd,num_bins,facecolor='black',alpha=0.5,label='Shuffled Observed vs Observed')
n1,bins1,patches1=plt.hist(jsd_ml,num_bins,facecolor='green',alpha=0.5,label='Average Observed vs Observed')
n1,bins1,patches1=plt.hist(jsd_rep,num_bins,facecolor='red',alpha=0.5,label='Pseudoreplicate performance')
plt.xlabel('Jensen Shannon Distance Profile Labels and Predictions in Probability Space')
plt.title("JSD Distribution ")
plt.legend(loc='best')
plt.savefig("~/chrombpnet/chrombpnet_use_snakefile/Treeshrew/TS_C8Epithelial_cells/output/predict/C8.1/C8.1.smooth.jsd.pdf",format='pdf',dpi=300, transparent=True)